# ChurnIQ v2.0 — Advanced Customer Churn Analysis

**End-to-End ML Pipeline**: Data Engineering → Feature Engineering → Advanced Modeling → Evaluation → Explainability → Business Impact

---

## Pipeline Overview
1. Data Loading & Advanced Preprocessing
2. Feature Engineering (20+ engineered features)
3. Exploratory Data Analysis
4. Model Training (5 models + Optuna tuning + SMOTE)
5. Comprehensive Evaluation (ROC, PR, Calibration, Business Cost)
6. SHAP Explainability
7. Customer Segmentation (KMeans)
8. Cohort Analysis
9. Time-based Validation

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
import logging

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

# Dark theme
plt.rcParams.update({
    'figure.facecolor': '#0F172A',
    'axes.facecolor': '#1E293B',
    'axes.edgecolor': '#334155',
    'axes.labelcolor': '#F8FAFC',
    'text.color': '#F8FAFC',
    'xtick.color': '#94A3B8',
    'ytick.color': '#94A3B8',
    'grid.color': '#334155',
    'grid.alpha': 0.3,
    'figure.figsize': (12, 6),
})

print('Environment ready.')

## 1. Data Loading & Preprocessing

In [ ]:
from src.preprocessing import load_and_clean_data, MissingValueHandler, OutlierHandler, CategoricalEncoder, FeatureScaler

df = load_and_clean_data('../data/telco_churn.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nChurn distribution:\n{df["Churn"].value_counts()}')
print(f'\nChurn rate: {df["Churn"].mean():.1%}')
print(f'\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head()

In [ ]:
# Handle missing values with multiple strategies
y = df['Churn']
X = df.drop(columns=['Churn'])

# Strategy comparison
for strategy in ['median', 'mean', 'knn']:
    handler = MissingValueHandler(strategy=strategy)
    X_imputed = handler.fit_transform(X)
    print(f'{strategy}: TotalCharges mean = {X_imputed["TotalCharges"].mean():.2f}, missing = {X_imputed.isnull().sum().sum()}')

# Use median (robust to outliers)
mv_handler = MissingValueHandler(strategy='median')
X = mv_handler.fit_transform(X)
print(f'\nFinal missing values: {X.isnull().sum().sum()}')

In [ ]:
# Outlier detection and treatment
outlier_handler = OutlierHandler(factor=1.5)

# Before
print('Before outlier treatment:')
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    print(f'  {col}: range [{X[col].min():.1f}, {X[col].max():.1f}]')

X = outlier_handler.fit_transform(X)

print('\nAfter outlier treatment:')
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    print(f'  {col}: range [{X[col].min():.1f}, {X[col].max():.1f}]')

## 2. Feature Engineering

In [ ]:
from src.feature_engineering import FeatureEngineer

fe = FeatureEngineer(n_segments=4)
X = fe.fit_transform(X)

print(f'Features after engineering: {X.shape[1]} (was {df.shape[1] - 1})')
print(f'\nNew features created:')
new_cols = [c for c in X.columns if c not in df.columns]
for col in new_cols:
    print(f'  - {col}: {X[col].dtype}')

X.head()

In [ ]:
# Engagement score distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Engagement score by churn
for churn_val, color, label in [(0, '#3B82F6', 'Retained'), (1, '#EF4444', 'Churned')]:
    axes[0].hist(X[y == churn_val]['engagement_score'], bins=20, alpha=0.7, color=color, label=label)
axes[0].set_title('Engagement Score by Churn', fontweight='bold')
axes[0].legend()

# Risk score by churn
for churn_val, color, label in [(0, '#3B82F6', 'Retained'), (1, '#EF4444', 'Churned')]:
    axes[1].hist(X[y == churn_val]['risk_score'], bins=20, alpha=0.7, color=color, label=label)
axes[1].set_title('Risk Score by Churn', fontweight='bold')
axes[1].legend()

# Behavior segments
seg_churn = pd.crosstab(X['behavior_segment'], y, normalize='index')
seg_churn.plot(kind='bar', stacked=True, ax=axes[2], color=['#3B82F6', '#EF4444'])
axes[2].set_title('Churn Rate by Behavior Segment', fontweight='bold')
axes[2].set_ylabel('Proportion')

plt.tight_layout()
plt.savefig('../models/plots/feature_engineering_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Encoding & Scaling

In [ ]:
# Encode categorical variables
encoder = CategoricalEncoder(method='onehot')
encoder.fit(X, y)
X_encoded = encoder.transform(X)

print(f'Shape after encoding: {X_encoded.shape}')
print(f'Columns: {list(X_encoded.columns)}')

# Scale features
scaler = FeatureScaler(method='robust')
X_scaled = scaler.fit_transform(X_encoded)

print(f'\nFinal feature matrix: {X_scaled.shape}')

## 4. Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from src.train import train_all_models, select_best_model, cross_validate_model, save_model

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {X_train.shape[0]} samples ({y_train.mean():.1%} churn)')
print(f'Test:  {X_test.shape[0]} samples ({y_test.mean():.1%} churn)')

In [ ]:
# Train all models with SMOTE and MLflow tracking
results = train_all_models(
    X_train, y_train, X_test, y_test,
    tune=True,         # Enable Optuna hyperparameter tuning
    n_trials=30,       # Number of Optuna trials
    use_smote=True,    # Handle class imbalance
    track_mlflow=True, # Track experiments in MLflow
)

# Display results
results_df = pd.DataFrame({
    name: info['metrics'] for name, info in results.items()
}).T.sort_values('roc_auc', ascending=False)

print('\nModel Comparison:')
results_df

In [ ]:
# Select best model
best_name, best = select_best_model(results)
print(f'Best model: {best_name}')
print(f'Metrics: {best["metrics"]}')

# Cross-validate
cv_results = cross_validate_model(best['model'], X_train, y_train, n_splits=5)
print(f'\nCross-Validation Results (5-Fold):')
for metric, vals in cv_results.items():
    print(f'  {metric}: {vals["mean"]:.4f} (+/- {vals["std"]:.4f})')

## 5. Comprehensive Evaluation

In [ ]:
from src.evaluate import ModelEvaluator
import os
os.makedirs('../models/plots', exist_ok=True)

evaluator = ModelEvaluator(output_dir='../models/plots')

eval_results = evaluator.full_evaluation(
    y_test, best['y_pred'], best['y_proba'], model_name=best_name
)

print('\nEvaluation Metrics:')
for k, v in eval_results['metrics'].items():
    print(f'  {k}: {v:.4f}')

print(f'\nOptimal threshold: {eval_results["optimal_threshold"]:.2f}')
print(f'\nBusiness Impact:')
for k, v in eval_results['business_impact'].items():
    print(f'  {k}: {v}')

In [ ]:
# Display all evaluation plots
from IPython.display import Image, display
import os

plot_dir = '../models/plots'
for plot_file in sorted(os.listdir(plot_dir)):
    if plot_file.endswith('.png'):
        print(f'\n--- {plot_file} ---')
        display(Image(filename=os.path.join(plot_dir, plot_file)))

## 6. SHAP Explainability

In [ ]:
# Extract the actual model from the pipeline for SHAP
actual_model = best['model']

# Global SHAP analysis
shap_values, explainer = evaluator.shap_analysis(
    actual_model, X_test, feature_names=list(X_test.columns), max_display=20
)

print(f'SHAP values shape: {shap_values.shape}')
print('\nTop 10 features by mean |SHAP|:')
mean_shap = np.abs(shap_values).mean(axis=0)
top_features = sorted(zip(X_test.columns, mean_shap), key=lambda x: x[1], reverse=True)[:10]
for feat, val in top_features:
    print(f'  {feat}: {val:.4f}')

In [ ]:
# Local explanation: explain a high-risk customer
import shap

high_risk_idx = best['y_proba'].argmax()
print(f'Explaining customer #{high_risk_idx} (churn prob: {best["y_proba"][high_risk_idx]:.3f})')

explanation = evaluator.explain_single_prediction(
    explainer, X_test.iloc[[high_risk_idx]], feature_names=list(X_test.columns)
)

# Waterfall plot
fig = plt.figure(figsize=(12, 8))
shap.plots.waterfall(explanation, show=False)
plt.title('SHAP Waterfall — High Risk Customer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/plots/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Customer Segmentation (KMeans)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Reload original data for segmentation
df_seg = load_and_clean_data('../data/telco_churn.csv')
seg_features = df_seg[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
seg_features = seg_features.fillna(seg_features.median())

scaler_seg = StandardScaler()
seg_scaled = scaler_seg.fit_transform(seg_features)

# Elbow method
inertias = []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(seg_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(K_range, inertias, 'o-', color='#EF4444', linewidth=2, markersize=8)
ax.set_xlabel('Number of Clusters (k)', fontsize=12)
ax.set_ylabel('Inertia', fontsize=12)
ax.set_title('Elbow Method for Optimal k', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../models/plots/elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fit optimal KMeans
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_seg['segment'] = kmeans.fit_predict(seg_scaled)

# Segment profiles
segment_profiles = df_seg.groupby('segment').agg({
    'tenure': 'mean',
    'MonthlyCharges': 'mean',
    'TotalCharges': 'mean',
    'Churn': ['mean', 'count'],
}).round(2)

segment_profiles.columns = ['Avg Tenure', 'Avg Monthly', 'Avg Total', 'Churn Rate', 'Size']
segment_names = {0: 'New & Low-spend', 1: 'Loyal & High-spend', 2: 'Mid-tenure & Mid-spend', 3: 'At-risk'}
segment_profiles.index = [segment_names.get(i, f'Segment {i}') for i in segment_profiles.index]

print('Customer Segments:')
segment_profiles

In [ ]:
# Visualize segments
fig = px.scatter_3d(
    df_seg, x='tenure', y='MonthlyCharges', z='TotalCharges',
    color=df_seg['segment'].astype(str),
    opacity=0.6,
    title='Customer Segments (3D)',
    labels={'color': 'Segment'},
)
fig.update_layout(
    paper_bgcolor='#0F172A', font_color='#F8FAFC',
    scene=dict(
        xaxis=dict(backgroundcolor='#1E293B'),
        yaxis=dict(backgroundcolor='#1E293B'),
        zaxis=dict(backgroundcolor='#1E293B'),
    ),
)
fig.show()

## 8. Cohort Analysis

In [ ]:
# Cohort analysis by tenure groups
df_cohort = load_and_clean_data('../data/telco_churn.csv')
df_cohort['tenure_cohort'] = pd.cut(
    df_cohort['tenure'],
    bins=[0, 6, 12, 24, 36, 48, 60, 72],
    labels=['0-6m', '7-12m', '13-24m', '25-36m', '37-48m', '49-60m', '61-72m'],
    include_lowest=True,
)

# Churn rate by cohort
cohort_churn = df_cohort.groupby('tenure_cohort').agg({
    'Churn': ['mean', 'count'],
    'MonthlyCharges': 'mean',
    'TotalCharges': 'mean',
}).round(3)
cohort_churn.columns = ['Churn Rate', 'Size', 'Avg Monthly', 'Avg Total']

print('Cohort Analysis:')
cohort_churn

In [ ]:
# Cohort visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Churn rate by cohort
colors = ['#EF4444' if r > 0.25 else '#F59E0B' if r > 0.15 else '#10B981' 
          for r in cohort_churn['Churn Rate']]
axes[0].bar(cohort_churn.index, cohort_churn['Churn Rate'] * 100, color=colors, edgecolor='#1E293B')
axes[0].set_title('Churn Rate by Tenure Cohort', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)', fontsize=12)
axes[0].set_xlabel('Tenure Cohort', fontsize=12)
for i, v in enumerate(cohort_churn['Churn Rate']):
    axes[0].text(i, v * 100 + 0.5, f'{v:.1%}', ha='center', fontsize=9, fontweight='bold')

# Cohort size
axes[1].bar(cohort_churn.index, cohort_churn['Size'], color='#3B82F6', edgecolor='#1E293B')
axes[1].set_title('Cohort Size', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Customers', fontsize=12)
axes[1].set_xlabel('Tenure Cohort', fontsize=12)

plt.tight_layout()
plt.savefig('../models/plots/cohort_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Time-Based Validation

In [ ]:
# Simulate time-based validation using tenure as a proxy for time
# Train on customers who joined earlier (higher tenure), test on newer customers
from sklearn.metrics import roc_auc_score, f1_score

df_time = load_and_clean_data('../data/telco_churn.csv')
fe_time = FeatureEngineer(n_segments=4)

y_time = df_time['Churn']
X_time = df_time.drop(columns=['Churn'])
X_time = fe_time.fit_transform(X_time)

encoder_time = CategoricalEncoder(method='onehot')
encoder_time.fit(X_time, y_time)
X_time = encoder_time.transform(X_time)

mv_time = MissingValueHandler(strategy='median')
X_time = mv_time.fit_transform(X_time)

scaler_time = FeatureScaler(method='robust')
X_time = scaler_time.fit_transform(X_time)

# Add tenure back for splitting
X_time['_tenure'] = df_time['tenure'].values

# Split: train on long-tenure (established), test on short-tenure (new)
median_tenure = df_time['tenure'].median()
train_mask = X_time['_tenure'] >= median_tenure
test_mask = X_time['_tenure'] < median_tenure

X_train_time = X_time[train_mask].drop(columns=['_tenure'])
y_train_time = y_time[train_mask]
X_test_time = X_time[test_mask].drop(columns=['_tenure'])
y_test_time = y_time[test_mask]

print(f'Time-based split:')
print(f'  Train (tenure >= {median_tenure}): {len(X_train_time)} ({y_train_time.mean():.1%} churn)')
print(f'  Test  (tenure <  {median_tenure}): {len(X_test_time)} ({y_test_time.mean():.1%} churn)')

In [ ]:
import xgboost as xgb

# Train on established customers, evaluate on new customers
model_time = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=3, eval_metric='logloss', random_state=42,
)
model_time.fit(X_train_time, y_train_time)

y_proba_time = model_time.predict_proba(X_test_time)[:, 1]
y_pred_time = (y_proba_time >= 0.5).astype(int)

auc_time = roc_auc_score(y_test_time, y_proba_time)
f1_time = f1_score(y_test_time, y_pred_time)

print(f'Time-Based Validation Results:')
print(f'  ROC-AUC: {auc_time:.4f}')
print(f'  F1 Score: {f1_time:.4f}')
print(f'\nNote: This simulates predicting churn for NEW customers using patterns')
print(f'learned from ESTABLISHED customers — closer to real-world deployment.')

## 10. Save Artifacts

In [ ]:
import joblib

# Save best model
metadata = {
    'model_name': best_name,
    'metrics': best['metrics'],
    'cv_results': {k: v['mean'] for k, v in cv_results.items()},
    'feature_names': list(X_train.columns),
}
save_model(best['pipeline'], metadata)

# Save preprocessing artifacts
artifacts = {
    'feature_engineer': fe,
    'encoder': encoder,
    'missing_handler': mv_handler,
    'scaler': scaler,
}
joblib.dump(artifacts, '../models/preprocessing_artifacts.joblib')

print('All artifacts saved!')
print(f'  Model: models/best_model.joblib')
print(f'  Preprocessing: models/preprocessing_artifacts.joblib')
print(f'  Metrics: models/results.json')

## Summary

### Key Results
- **Best Model**: Selected via ROC-AUC comparison across 5 models
- **Class Imbalance**: Handled with SMOTE (synthetic minority oversampling)
- **Hyperparameter Tuning**: Optuna Bayesian optimization (30 trials)
- **Cross-Validation**: 5-fold stratified CV for robust estimates
- **Explainability**: SHAP values for global + local explanations
- **Business Impact**: Cost-benefit analysis quantifying model ROI

### Top Churn Drivers (from SHAP)
1. Contract type (month-to-month = highest risk)
2. Tenure (new customers churn most)
3. Monthly charges (higher charges = higher risk)
4. Internet service type (fiber optic = highest risk)
5. Payment method (electronic check = highest risk)

### Business Recommendations
1. **Priority 1**: Incentivize long-term contracts (15-20% discount)
2. **Priority 1**: Structured onboarding for first 90 days
3. **Priority 2**: Investigate fiber optic service quality
4. **Priority 2**: Migrate customers to auto-pay methods
5. **Priority 3**: Monthly churn scoring + proactive outreach